# Исследование Classification Schemes

У `research-output` в схеме есть поле `category` и `type`.
Оба определяются как тип `WSClassification`.
Используя инструменты, полученные при анализе `research-outputs.json`, проведем похожий анализ и для `classification-schemes.json`.

## Импорт зависимостей

In [1]:
# imports
import os
import json
import pandas as pd

In [2]:
from notebooks import utils

## Чтение выборки

In [3]:
classification_data = utils.load_from_json('../data/classification-schemes.json')['items']

## Чтение спецификации

In [4]:
SWAGGER_SPEC_URL = 'https://pure.spbu.ru/ws/api/522/swagger.json'
api_spec = utils.fetch_api_swagger_json(SWAGGER_SPEC_URL)

Fetched successfully!


In [5]:
api_spec['definitions']['WSClassification']['properties']

{'pureId': {'type': 'integer', 'format': 'int64', 'xml': {'attribute': True}},
 'externalId': {'type': 'string', 'xml': {'attribute': True}},
 'externalIdSource': {'type': 'string', 'xml': {'attribute': True}},
 'externallyManaged': {'type': 'boolean', 'xml': {'attribute': True}},
 'uri': {'type': 'string', 'xml': {'attribute': True}},
 'term': {'$ref': '#/definitions/WSLocalizedString'},
 'disabled': {'type': 'boolean', 'xml': {'attribute': True}},
 'description': {'$ref': '#/definitions/WSLocalizedString'},
 'classificationRelations': {'type': 'array',
  'xml': {'wrapped': True},
  'items': {'xml': {'name': 'classificationRelation'},
   '$ref': '#/definitions/WSClassificationRelation'}}}

Чтобы не вызывать все методы в деталях, напишем функцию, которая соберет все util-методы для аналитики в одном месте.

In [6]:
def generate_frequences_report(spec, schema_name, items):
    schema_df = utils.fetch_schema(spec, schema_name)
    
    known_frequences, unknown_frequences = utils.count_field_frequences(
        items = items, 
        keys = list(schema_df['field'])
    )

    known_frequences_df = pd.DataFrame(known_frequences.items(), columns=['field', 'count'])
    unknown_frequences_df = pd.DataFrame(unknown_frequences.items(), columns=['field', 'count'])

    schema_fields_freqs = pd.merge(
        left=schema_df, 
        right=known_frequences_df, 
        on='field', 
        how='outer'
    ).sort_values(
        by='count',
        ascending=False
    )

    return schema_fields_freqs, unknown_frequences_df

In [7]:
schema_fields_freqs, unknown_fields_freqs = generate_frequences_report(
    api_spec,
    'WSClassification',
    classification_data
)

In [8]:
unknown_fields_freqs

,field,count
0,uuid,315
1,baseUri,315
2,info,315
3,typeClassification,264
4,containedClassifications,281
5,associatedSchemes,1


## Парсинг атрибутов в выборке

Попробуем уложить классификации в pandas dataframe, чтобы уловить зависимости.
Будем искать `uri`, `uuid`, `description`, `containedClassifications`.

Из `containedClassifications` попробуем также вытащить `uri`, чтобы понять семантику.

In [ ]:
def parse_classifications(classification_data):
    result =[]

    for classification in classification_data:
        parsed_object = dict()
        parsed_object['pure_id'] = classification['pureId']
        parsed_object['uuid'] = classification['uuid']

        # get base uri for root entry
        base_uri = classification['baseUri']
        parsed_object['uri'] = base_uri

        # find description
        if 'description' in classification.keys():
            localized_descriptions = classification['description']['text']
            parsed_object['description'] = localized_descriptions[0]['value']
        else:
            parsed_object['description'] = 'None'

        if 'containedClassifications' in classification.keys():
            contained_classifications = classification['containedClassifications']
            contained_classifications_ids = dict()

            for contained_classification in contained_classifications:
                cc_pure_id = contained_classification['pureId']
                cc_uri = contained_classification['uri'].replace(base_uri, '')
                contained_classifications_ids[cc_pure_id] = cc_uri
            
            parsed_object['contained_classifications_ids'] = contained_classifications_ids
        else:
            parsed_object['contained_classifications_ids'] = {}
        
        result.append(parsed_object)
    
    return result

In [10]:
parsed_classification = parse_classifications(classification_data)
parsed_classification_df = pd.DataFrame(parsed_classification)
parsed_classification_df.sort_values(by='pure_id').head()

,pure_id,uuid,uri,description,contained_classifications_ids
0,103,0375dc3c-a4e9-4ed4-8f88-3793594e4543,/dk/atira/pure/core/classificationschemes,Types for classification schemes,"{105: '/taxonomi', 107: '/misc', 109: '/role',..."
1,118,7c7b849b-ac53-47fb-bcbb-d0a3fb58abfe,/dk/atira/pure/organisation/namevariants,Organisation name variants,"{120: '/shortname', 122: '/sortname', 124: '/w..."
2,129,989ffa0b-ace5-42d8-a9f5-30e2f559af7f,/dk/atira/pure/organisation/organisationaddres...,Address types,{17328: '/visiting_address'}
3,135,9f7b3f0c-dd07-4484-b821-a56f4513b84d,/dk/atira/pure/organisation/organisationphonen...,Phone number types,"{137: '/phone', 139: '/fax', 141: '/mobile', 1..."
4,145,742de1ad-ab79-4dc9-95ac-20898a14442a,/dk/atira/pure/organisation/organisationwebadd...,Web address types,{17322: '/official_website'}


Можно попробовать поискать записи, которые хоть немного связаны с научной деятельностью

In [11]:
parsed_classification_df[parsed_classification_df['uri'].str.contains('research', case = False)]

,pure_id,uuid,uri,description,contained_classifications_ids
30,919,6b8367bf-8b1f-44cf-9b39-6e619c1599a8,/dk/atira/pure/links/researchoutput,Types of links for Research output,"{921: '/unspecified', 12683: '/scopuspublicati..."
35,947,7befd009-b675-4947-a2ca-f857a8eb3006,/dk/atira/pure/researchoutput/status,Publication state,"{949: '/inprep', 951: '/submitted', 953: '/inp..."
64,2106,5e61f0db-1a5c-43ac-a19d-aa2f8db57997,/dk/atira/pure/association/bidirectional/resea...,Research output associations,"{4258: '/predecessor', 4261: '/successor'}"
68,2696,d6514cb5-0b5f-40f6-87fb-6fd4ed8fd309,/dk/atira/pure/researchoutput/openaccesspermis...,Types of open access states for publications,"{2698: '/open', 2700: '/embargoed', 2702: '/cl..."
85,2816,ab877d40-99dc-4cc8-b3d6-086bc2373f53,/dk/atira/pure/researchoutputref2014/researcho...,Research Output REF2014 types,"{2818: '/researchoutputref2014', 2821: '/resea..."
86,2825,aa4a3e1a-c2d2-4d8c-b839-00f6a3b4df4f,/dk/atira/pure/researchoutput/researchoutputtypes,Types of research output,"{2827: '/contributiontobookanthology', 3949: '..."
89,2870,9ba6a38c-301e-4dd8-9a9b-45405d5149e4,/dk/atira/pure/equipment/research_technique,Equipment research technique,{16844: '/other'}
100,2964,8f4d1692-0112-407d-95ce-a52bf3aaffd1,/dk/atira/pure/ref2014/researchoutputref2014/o...,REF2014 REF2 output types,"{2966: '/a', 2968: '/b', 2970: '/c', 2972: '/d..."
114,3923,82d8b6b9-812d-4871-afcb-9559282ab6e9,/dk/atira/pure/researchoutput/peerreviewable,Peer review for publication type,"{3925: '/peerreviewable', 3927: '/notpeerrevie..."
115,3935,f1ce2bfe-779c-48b6-8e0e-6f079056cdb3,/dk/atira/pure/researchoutput/category,Publication category,"{3937: '/research', 3941: '/popular', 3943: '/..."


## Результаты

Интересными группами могут быть `/dk/atira/pure/researchoutput/researchoutputtypes` и `/dk/atira/pure/links/researchoutput`. При определении связей с людьми, полезным может быть `/dk/atira/pure/researchoutput/role/*`.

Посмотрим на записи с категориями и типами научной деятельности.

In [12]:
def print_contained_classifications_info(dataframe, uri):
    row = dataframe[dataframe['uri'] == uri].index[0]
    dict_entries = dataframe.loc[row, 'contained_classifications_ids']

    print(f"Total elements: {len(dict_entries)}")
    for key, value in dict_entries.items():
        print(f"{key}: {value}")

По всей видимости, `category` - более общая группа для научной деятельности.

In [13]:
print_contained_classifications_info(
    dataframe=parsed_classification_df,
    uri='/dk/atira/pure/researchoutput/category',
)

Total elements: 6
3937: /research
3941: /popular
3943: /other
3939: /education
19236878: /transfer
19236880: /communication


`type` более точно определяет тип деятельности:

In [14]:
print_contained_classifications_info(
    dataframe=parsed_classification_df, 
    uri='/dk/atira/pure/researchoutput/researchoutputtypes'
)

Total elements: 77
2827: /contributiontobookanthology
3949: /contributiontojournal
3959: /bookanthology
3953: /contributiontoperiodical
3961: /workingpaper
3951: /contributiontoconference
3963: /nontextual
3965: /thesis
3955: /patent
3967: /memorandum
3969: /contributiontomemorandum
3957: /othercontribution
3973: /contributiontojournal/article
3977: /contributiontojournal/letter
3981: /contributiontojournal/comment
3985: /contributiontojournal/book
3989: /contributiontojournal/scientific
3993: /contributiontojournal/systematicreview
3997: /contributiontojournal/shortsurvey
4001: /contributiontojournal/editorial
4005: /contributiontojournal/abstract
4011: /contributiontobookanthology/article
4015: /contributiontobookanthology/chapter
4022: /contributiontobookanthology/entry
4029: /contributiontobookanthology/conference
4036: /contributiontobookanthology/foreword
4043: /contributiontobookanthology/other
4052: /contributiontoconference/paper
4056: /contributiontoconference/poster
4060: /c

Посмотрим на запись со ссылками:

In [15]:
print_contained_classifications_info(
    dataframe=parsed_classification_df,
    uri='/dk/atira/pure/links/researchoutput'
)

Total elements: 4
921: /unspecified
12683: /scopuspublication
12685: /scopuscitations
64558712: /portalmultimedia
